# Survey Measurement Validation

This historical analysis compares customer-experience measures collected through two independent survey sources for a restaurant brand.

## Analytical workflow

- Harmonize differently coded response scales
- Align overall satisfaction, revisit intent, and recommendation measures
- Compare pooled and wave-level samples
- Evaluate distributional similarity with two-sample Kolmogorov–Smirnov tests
- Evaluate mean differences with Welch’s independent-samples t-tests

## Data availability

The original cleaned survey files are not included. Source labels and brand references have been generalized for public presentation.

## Interpretation note

A non-significant test result does not prove that samples are equivalent. It indicates that the analysis did not detect a difference at the selected significance level. Formal equivalence testing would be required to support a claim of practical equivalence.


In [ ]:
# Import libraries necessary for this project
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display # Allows the use of display() for DataFrames

# mobile_1 = pd.read_csv("Mobile Sample Data/Mobile Sample - Restaurant Brand Restaurants - StoreExperienceTracker1of3April18 - Data.csv", low_memory=False)
# mobile_2 = pd.read_csv("Mobile Sample Data/Mobile Sample - Restaurant Brand Restaurants - StoreExperienceTracker2of3July18 - Merged Data.csv", low_memory=False)
# mobile_3 = pd.read_csv("Mobile Sample Data/Mobile Sample - Restaurant Brand Restaurants - StoreExperienceTracker3of3October18 - Merged Data (Working).csv", low_memory=False)
#c1 = pd.read_csv("Benchmark Sample Data/Detail Report DMA (Updated) (1).csv", encoding = "ISO-8859-1", low_memory=False)
c2 = pd.read_csv("SimilarityTesting/Benchmark Sample Data/Detail Report DMA (Updated) (2).csv", encoding = "ISO-8859-1", low_memory=False)
#c3 = pd.read_csv("Benchmark Sample Data/Detail Report DMA (Updated) (3).csv", encoding = "ISO-8859-1", low_memory=False)
c1 = pd.read_csv("./SimilarityTesting/AugustData/Benchmark Sample/Restaurant Brand - Detail Data Report (01012019-01312019).csv",encoding="ISO-8859-1",sep="|")


# print("Mobile Sample wave 1 dataset has {} samples with {} features each.".format(*mobile_1.shape))
# print("Mobile Sample wave 2 dataset has {} samples with {} features each.".format(*mobile_2.shape))
# print("Mobile Sample wave 3 dataset has {} samples with {} features each.".format(*mobile_3.shape))
print("Benchmark Sample wave 1 dataset has {} samples with {} features each.".format(*c1.shape))
print("Benchmark Sample wave 1 dataset has {} samples with {} features each.".format(*c2.shape))
#print("Benchmark Sample wave 1 dataset has {} samples with {} features each.".format(*c3.shape))
#df.columns = ['pid', 'name', 'arrival', 'dwell', 'confidence', 'date']
print("Mfour wave 1 data view below")

In [ ]:
c1.head()

In [ ]:
c2.head()

In [ ]:
print("Mobile Sample variables:")
print("Note: not showing list due to size")
#list(mobile_1)

In [ ]:
print("Benchmark Sample variables:")
print("Note: not showing list due to size")
#list(c1)

# Unify and align priority questions
Data prep to allow for analysis by aligning questions, remove NAs, etc

In [ ]:
val1 = {'Strongly Agree':5, 
      'Agree':4,
      'Neither Agree Nor Disagree':3,
      'Disagree':2,
      'Strongly Disagree':1,}

val2 = {'Extremely Likely':5, 
      'Likely':4,
      'Neutral':3,
      'Unlikely':2,
      'Extremely Unlikely':1,}

val3 = {'Extremely Likely':11,
        '9':10,
        '8':9,
        '7':8,
        '6':7,
        '5':6,
        '4':5,
        '3':4,
        '2':3,
        '1':2,
      'Extremely Unlikely ':1,}

###

c1['Overall Satisfaction'] = c1['Overall Satisfaction'].replace(to_replace=val1)

c1['Revisit'] = c1['Revisit'].replace(to_replace=val2)

c1['Recommend'] = c1['Recommend'].replace(to_replace=val3)

###


c2['Overall Satisfaction'] = c2['Overall Satisfaction'].replace(to_replace=val1)

c2['Revisit'] = c2['Revisit'].replace(to_replace=val2)

c2['Recommend'] = c2['Recommend'].replace(to_replace=val3)

###


c3['Overall Satisfaction'] = c3['Overall Satisfaction'].replace(to_replace=val1)

c3['Revisit'] = c3['Revisit'].replace(to_replace=val2)

c3['Recommend'] = c3['Recommend'].replace(to_replace=val3)

In [ ]:
print("Benchmark Sample OSAT Data Variables and Distribution")
c1['Overall Satisfaction'].value_counts(dropna=False)

In [ ]:
#print(c1['Overall Satisfaction'].unique())

In [ ]:
print("Benchmark Sample Revisit Data Variables and Distribution")
c1['Revisit'].value_counts(dropna=False)

In [ ]:
#print(c1['Revisit'].unique())

In [ ]:
print("Benchmark Sample Recommend Data Variables and Distribution")
c1['Recommend'].value_counts(dropna=False)

In [ ]:
#extremely likely=10 ... extremely unlikely=0
#print(c1['Recommend'].unique())

In [ ]:
#OSAT
#strongly agree=5 ... strongly disagree=1
#print(mobile_1['Q3'].unique())

In [ ]:
#revisit
#extremely likely=5 ... extremely unlikely=1
#print(mobile_1['Q4'].unique())

In [ ]:
#NPS
#1=low ... 10=high
#print(mobile_1['Q5'].unique())

In [ ]:
#Mobile Sample data
w1=mobile_1[['Q3','Q4','Q5']].copy()
w2=mobile_2[['Q3','Q4','Q5']].copy()
w3=mobile_3[['Q3','Q4','Q5']].copy()

concatw=pd.concat([w1,w2,w3])
concatw=concatw.dropna()
print("Mobile Sample 3 waves cleaned and merged:", len(concatw['Q3']))
concatw.head()

In [ ]:
x1=c1[['Overall Satisfaction','Revisit','Recommend']].copy()
x2=c2[['Overall Satisfaction','Revisit','Recommend']].copy()
x3=c3[['Overall Satisfaction','Revisit','Recommend']].copy()

concatx=pd.concat([x1,x2,x3])
concatx=concatx.dropna()
concatx.columns = ['OSAT', 'RPI', 'NPS']
print("Benchmark Sample 3 waves cleaned and merged:", len(concatx['OSAT']))
concatx.head()

# Stats Merged Waves Data
#a 2 sample KS test to compare distribution
#b 2 sample t test to compare means

In [ ]:
from scipy import stats
from scipy.stats import kstest
from scipy.stats import ks_2samp
from scipy.stats import ttest_ind
import matplotlib.pyplot as plot

In [ ]:
print("Mobile Sample merged data description")
concatw.describe()

In [ ]:
print("Benchmark Sample merged data description")
concatx.describe()

Test homogeniety of variance

In [ ]:
#test homogeniety of variance, if variance is significant than use welche's test, otherwise use t-test
print("OSAT")
stats.levene(concatw['Q3'], concatx['OSAT'])

In [ ]:
print("RPI")
stats.levene(concatw['Q4'], concatx['RPI'])

In [ ]:
print("NPS")
stats.levene(concatw['Q5'], concatx['NPS'])

since the pvals<.05 the variances are not homogenious
***

In [ ]:
#check for normality - although dealing with different sample lengths so ...
#diff = (concatw['Q3'] - concatx['OSAT'])
#will test individually
print("testing for normality:")
print("Mobile Sample Q3:", stats.shapiro(concatw['Q3']))
print("Benchmark Sample OSAT:", stats.shapiro(concatx['OSAT']))
print("Mobile Sample Q4:", stats.shapiro(concatw['Q4']))
print("Benchmark Sample RPI:", stats.shapiro(concatx['RPI']))
print("Mobile Sample Q5:", stats.shapiro(concatw['Q5']))
print("Benchmark Sample NPS:", stats.shapiro(concatx['NPS']))

print("Note: for Benchmark Sample data sets with larger sample sizes, need to use KS - occuring below")

KS (Kolmogorov–Smirnov test) test checks to see if the distribution is similar

In [ ]:
#KS (Kolmogorov–Smirnov test) test checks to see if the distribution is similar, in this case against a normal distribution
print("KS test, Mobile Sample vs norm")
kstest(concatw['Q3'], 'norm')

In [ ]:
print("KS test, Benchmark Sample vs norm")
kstest(concatx['OSAT'], 'norm')

Using both Shapiro-Wilks and Kolmogorov-Smirnov to test normality, neither Mobile Sample or Benchmark Sample is normally distributed data

In [ ]:
print("KS test, compare Mobile Sample vs Benchmark Sample for OSAT")
ks_2samp(concatw['Q3'], concatx['OSAT'])

In [ ]:
print("Mobile Sample OSAT")
concatw.hist(column='Q3')
plt.show()

In [ ]:
print("Benchmark Sample OSAT")
concatx.hist(column='OSAT')
plt.show()

In [ ]:
print("KS test, compare Mobile Sample vs Benchmark Sample for RPI")
ks_2samp(concatw['Q4'], concatx['RPI'])

In [ ]:
print("Mobile Sample RPI")
concatw.hist(column='Q4')
plt.show()

In [ ]:
print("Benchmark Sample RPI")
concatx.hist(column='RPI')
plt.show()

In [ ]:
print("KS test, compare Mobile Sample vs Benchmark Sample for NPS")
ks_2samp(concatw['Q5'], concatx['NPS'])

In [ ]:
print("Mobile Sample NPS")
concatw.hist(column='Q5')
plt.show()

In [ ]:
print("Benchmark Sample NPS")
concatx.hist(column='NPS')
plt.show()

For all, as pvals<.05 there is significance between distributions of Mobile Sample and Benchmark Sample data
******

Compare means

In [ ]:
print("Mobile Sample vs Benchmark Sample OSAT")
ttest_ind(concatw['Q3'], concatx['OSAT'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(concatw['Q3'], concatx['OSAT'])

due to pvalue>.05, there is no significant difference, overall satisfaction the test did not detect a statistically significant mean difference with merged data

In [ ]:
print("Mobile Sample vs Benchmark Sample RPI")
ttest_ind(concatw['Q4'], concatx['RPI'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(concatw['Q4'], concatx['RPI'])

In [ ]:
print("Mobile Sample vs Benchmark Sample NPS")
ttest_ind(concatw['Q5'], concatx['NPS'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(concatw['Q5'], concatx['NPS'])

RPI and NPS means had pvalues<.05 and therefore aren't similar with merged data
***

# Run waves data individually

In [ ]:
print("Mobile Sample wave 1")
w1.describe()

In [ ]:
print("Benchmark Sample wave 1")
x1.columns = ['OSAT', 'RPI', 'NPS']
x1=x1.dropna()
x1.describe()

In [ ]:
#print("Mobile Sample wave 1 OSAT")
#w1['Q3'].value_counts(dropna=False)

In [ ]:
#print("Benchmark Sample wave 1 OSAT")
#x1=x1.dropna()
#x1['OSAT'].value_counts(dropna=False)

In [ ]:
print("KS test to see distribution of wave 1 data for OSAT between Mobile Sample and Benchmark Sample")
ks_2samp(w1['Q3'], x1['OSAT'])

due to low pvalue, the test detected a statistically significant distribution difference

In [ ]:
print("Mobile Sample wave 1 OSAT")
w1.hist(column='Q3')
plt.show()

In [ ]:
print("Benchmark Sample wave 1 OSAT")
x1.hist(column='OSAT')
plt.show()

In [ ]:
print("Test means of wave 1 data for OSAT between Mobile Sample and Benchmark Sample")
ttest_ind(w1['Q3'], x1['OSAT'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(w1['Q3'], x1['OSAT'])

due to pvalue, the test did not detect a statistically significant mean difference

In [ ]:
print("KS test to see distribution of wave 1 data for RPI between Mobile Sample and Benchmark Sample")
ks_2samp(w1['Q4'], x1['RPI'])

In [ ]:
print("Mobile Sample wave 1 RPI")
w1.hist(column='Q4')
plt.show()

In [ ]:
print("Benchmark Sample wave 1 RPI")
x1.hist(column='RPI')
plt.show()

In [ ]:
print("Test means of wave 1 data for RPI between Mobile Sample and Benchmark Sample")
ttest_ind(w1['Q4'], x1['RPI'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(w1['Q4'], x1['RPI'])

due to pvalue, the test detected a statistically significant mean difference

In [ ]:
print("KS test to see distribution of wave 1 data for NPS between Mobile Sample and Benchmark Sample")
ks_2samp(w1['Q5'], x1['NPS'])

In [ ]:
print("Mobile Sample wave 1 NPS")
w1.hist(column='Q5')
plt.show()

In [ ]:
print("Benchmark Sample wave 1 NPS")
x1.hist(column='NPS')
plt.show()

In [ ]:
print("Test means of wave 1 data for NPS between Mobile Sample and Benchmark Sample")
ttest_ind(w1['Q5'], x1['NPS'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(w1['Q5'], x1['NPS'])

due to pvalue, the test detected a statistically significant mean difference

In [ ]:
print("Mobile Sample wave 2")
w2.describe()

In [ ]:
print("Benchmark Sample wave 2")
x2.columns = ['OSAT', 'RPI', 'NPS']
x2=x2.dropna()
x2.describe()

In [ ]:
print("KS test to see distribution of wave 2 data for OSAT between Mobile Sample and Benchmark Sample")
ks_2samp(w2['Q3'], x2['OSAT'])

the test detected a statistically significant distribution difference

In [ ]:
print("Mobile Sample wave 2 OSAT")
w2.hist(column='Q3')
plt.show()

In [ ]:
print("Benchmark Sample wave 2 OSAT")
x2.hist(column='OSAT')
plt.show()

In [ ]:
print("Test means of wave 2 data for OSAT between Mobile Sample and Benchmark Sample")
ttest_ind(w2['Q3'], x2['OSAT'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(w2['Q3'], x2['OSAT'])

the test did not detect a statistically significant mean difference

In [ ]:
print("KS test to see distribution of wave 2 data for RPI between Mobile Sample and Benchmark Sample")
ks_2samp(w2['Q4'], x2['RPI'])

the test detected a statistically significant distribution difference

In [ ]:
print("Mobile Sample wave 2 RPI")
w2.hist(column='Q4')
plt.show()

In [ ]:
print("Benchmark Sample wave 2 RPI")
x2.hist(column='RPI')
plt.show()

In [ ]:
print("Test means of wave 2 data for RPI between Mobile Sample and Benchmark Sample")
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(w2['Q4'], x2['RPI'])

the test did not detect a statistically significant mean difference

In [ ]:
print("KS test to see distribution of wave 2 data for NPS between Mobile Sample and Benchmark Sample")
ks_2samp(w2['Q5'], x2['NPS'])

the test detected a statistically significant distribution difference

In [ ]:
print("Mobile Sample wave 2 NPS")
w2.hist(column='Q5')
plt.show()

In [ ]:
print("Benchmark Sample wave 2 NPS")
x2.hist(column='NPS')
plt.show()

In [ ]:
print("Test means of wave 2 data for NPS between Mobile Sample and Benchmark Sample")
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(w2['Q5'], x2['NPS'])

the test detected a statistically significant mean difference

In [ ]:
print("Mobile Sample wave 3")
w3.describe()

In [ ]:
print("Benchmark Sample wave 3")
x3.columns = ['OSAT', 'RPI', 'NPS']
x3=x3.dropna()
x3.describe()

In [ ]:
print("KS test to see distribution of wave 3 data for OSAT between Mobile Sample and Benchmark Sample")
ks_2samp(w3['Q3'], x3['OSAT'])

In [ ]:
print("Mobile Sample wave 3 OSAT")
w3.hist(column='Q3')
plt.show()

In [ ]:
print("Benchmark Sample wave 3 OSAT")
x3.hist(column='OSAT')
plt.show()

In [ ]:
print("Test means of wave 3 data for OSAT between Mobile Sample and Benchmark Sample")
ttest_ind(w3['Q3'], x3['OSAT'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(w3['Q3'], x3['OSAT'])

the test detected a statistically significant mean difference

In [ ]:
print("KS test to see distribution of wave 3 data for RPI between Mobile Sample and Benchmark Sample")
ks_2samp(w3['Q4'], x3['RPI'])

the test detected a statistically significant distribution difference

In [ ]:
print("Mobile Sample wave 3 RPI")
w3.hist(column='Q4')
plt.show()

In [ ]:
print("Benchmark Sample wave 3 RPI")
x3.hist(column='RPI')
plt.show()

In [ ]:
print("Test means of wave 3 data for RPI between Mobile Sample and Benchmark Sample")
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(w3['Q4'], x3['RPI'])

the test did not detect a statistically significant mean difference

In [ ]:
print("KS test to see distribution of wave 3 data for NPS between Mobile Sample and Benchmark Sample")
ks_2samp(w3['Q5'], x3['NPS'])

the test detected a statistically significant distribution difference

In [ ]:
print("Mobile Sample wave 3 NPS")
w3.hist(column='Q5')
plt.show()

In [ ]:
print("Benchmark Sample wave 3 NPS")
x3.hist(column='NPS')
plt.show()

In [ ]:
print("Test means of wave 3 data for NPS between Mobile Sample and Benchmark Sample")
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(w3['Q5'], x3['NPS'])

the test did not detect a statistically significant mean difference

# Combine Waves 2+3

In [ ]:
#Mobile Sample data
concatm2=pd.concat([w2,w3])
concatm2=concatm2.dropna()

print("Mobile Sample wave 2+3")
concatm2.describe()

In [ ]:
concatx2=pd.concat([x2,x3])
concatx2=concatx2.dropna()
concatx2.columns = ['OSAT', 'RPI', 'NPS']

print("Benchmark Sample wave 2+3")
concatx2.describe()

In [ ]:
print("KS test to see distribution of wave 1 data for OSAT between Mobile Sample and Benchmark Sample")
ks_2samp(concatm2['Q3'], concatx2['OSAT'])

the test detected a statistically significant distribution difference

In [ ]:
print("Mobile Sample wave 2+3 OSAT")
concatm2.hist(column='Q3')
plt.show()

In [ ]:
print("Benchmark Sample wave 2+3 OSAT")
concatx2.hist(column='OSAT')
plt.show()

In [ ]:
print("Test means of wave 1 data for OSAT between Mobile Sample and Benchmark Sample")
ttest_ind(concatm2['Q3'], concatx2['OSAT'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(concatm2['Q3'], concatx2['OSAT'])

the test detected a statistically significant mean difference

In [ ]:
print("KS test to see distribution of wave 1 data for OSAT between Mobile Sample and Benchmark Sample")
ks_2samp(concatm2['Q4'], concatx2['RPI'])

the test detected a statistically significant distribution difference

In [ ]:
print("Mobile Sample wave 2+3 OSAT")
concatm2.hist(column='Q4')
plt.show()

In [ ]:
print("Benchmark Sample wave 2+3 OSAT")
concatx2.hist(column='RPI')
plt.show()

In [ ]:
print("Test means of wave 1 data for OSAT between Mobile Sample and Benchmark Sample")
ttest_ind(concatm2['Q4'], concatx2['RPI'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(concatm2['Q4'], concatx2['RPI'])

the test detected a statistically significant mean difference

In [ ]:
print("KS test to see distribution of wave 1 data for OSAT between Mobile Sample and Benchmark Sample")
ks_2samp(concatm2['Q5'], concatx2['NPS'])

the test detected a statistically significant distribution difference

In [ ]:
print("Mobile Sample wave 2+3 OSAT")
concatm2.hist(column='Q5')
plt.show()

In [ ]:
print("Benchmark Sample wave 2+3 OSAT")
concatx2.hist(column='NPS')
plt.show()

In [ ]:
print("Test means of wave 1 data for OSAT between Mobile Sample and Benchmark Sample")
ttest_ind(concatm2['Q5'], concatx2['NPS'],equal_var = False)

In [ ]:
def welch_ttest(x, y): 
    ## Welch-Satterthwaite Degrees of Freedom ##
    dof = (x.var()/x.size + y.var()/y.size)**2 / ((x.var()/x.size)**2 / (x.size-1) + (y.var()/y.size)**2 / (y.size-1))
   
    t, p = stats.ttest_ind(x, y, equal_var = False)
    
    print("\n",
          f"Welch's t-test= {t:.4f}", "\n",
          f"p-value = {p:.4f}", "\n",
          f"Welch-Satterthwaite Degrees of Freedom= {dof:.4f}")

welch_ttest(concatm2['Q5'], concatx2['NPS'])

the test detected a statistically significant mean difference

## Wave 2+3 summary
1. Data the test detected a statistically significant distribution difference for OSAT, RPI, or NPS
2. Data means are not similar only for OSAT,RPI, or NPS
3. Data was checked as high as 95% confidence level (lower levels are more likely to find difference, not similarity)

# Summary:
Mobile Sample vs Benchmark Sample when 3 waves merged:
1. Data sets are not homogeneous for variance
2. Data the test detected a statistically significant distribution difference for OSAT, RPI, or NPS
3. Data the test did not detect a statistically significant mean difference only for OSAT, not similar for RPI or NPS

Mobile Sample vs Benchmark Sample data when 3 waves not merged:
1. Data the test detected a statistically significant distribution difference for Wave 1, 2, or 3 for any of OSAT, RPI, or NPS
2. Data the test did not detect a statistically significant mean difference for Wave 1 and 2 for OSAT, Wave 2 and 3 for RPI, and Wave 3 for NPS